# FD-IDS: Federated Learning + Knowledge Distillation cho IDS (DNN)

Bản dựng lại theo paper **FD-IDS** (Sensors 2025) trên dataset **CIC-IoT 2023** đã tiền xử lý.

**Kỹ thuật (bám sát paper):**
- Model: **DNN nhẹ** `[25 → 32 → 64 → 128 → 64 → 32 → 34]`, ReLU, softmax ở output.
- FL: **FedProx** (proximal term, μ=0.01) + **round-wise Knowledge Distillation** (global = teacher, local = student).
- Loss: `L = λ·CE + (1-λ)·KD_soft + β·L_proximal`  (λ=0.5, β=0.1, T=3).
- Aggregation: **weighted FedAvg** theo số mẫu client.

**Dataset:** 10 client non-IID (Dirichlet α=0.2), 25 feature đã chọn (XGBoost) + chuẩn hóa (QuantileTransformer), 34 lớp.
Dữ liệu CSV **đã chuẩn hóa sẵn** → notebook **không** chuẩn hóa lại, chỉ map nhãn qua `label_mapping.csv`.

**Hạ tầng:** Kaggle 2×T4, 10 client train song song trên 2 GPU (cân bằng tải LPT).

**Lịch train (đã chốt):** full data, `LOCAL_EPOCHS=1`, `COMMUNICATION_ROUNDS=10`.
Best model chọn theo **weighted-F1** (ưu tiên accuracy/weighted-F1, không xử lý mất cân bằng).

> Chạy lần lượt từng cell từ trên xuống. Cell smoke-test (model/loss/1-client) giúp bắt lỗi trước khi train full.

In [ ]:
# ===== Cell 1: Import, Config, Seed, Device, Output dirs =====
import os, json, time, random, gc, math
from pathlib import Path
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# --------- Đường dẫn dữ liệu (auto-detect nếu path mặc định không tồn tại) ---------
DATA_DIR = Path("/kaggle/input/datasets/odixe0502/data-for-10clients")
def _find_data_dir(default: Path) -> Path:
    if (default / "client_1_train.csv").exists():
        return default
    for base in [Path("/kaggle/input"), Path(".")]:
        if base.exists():
            hits = list(base.rglob("client_1_train.csv"))
            if hits:
                return hits[0].parent
    return default  # giữ nguyên; cell validate sẽ báo lỗi rõ ràng
DATA_DIR = _find_data_dir(DATA_DIR)

# --------- Hyperparameters (bám sát paper FD-IDS) ---------
CFG = SimpleNamespace(
    num_clients          = 10,
    input_dim            = 25,
    num_classes          = 34,
    hidden_dims          = (32, 64, 128, 64, 32),  # DNN paper
    dropout              = 0.0,                     # paper không dùng dropout
    local_epochs         = 1,
    comm_rounds          = 10,
    batch_size           = 4096,                    # DNN nhẹ + data lớn -> batch lớn cho throughput
    learning_rate        = 1e-3,                    # Adam, paper
    weight_decay         = 0.0,
    temperature          = 3.0,                     # T (KD)
    lambda_hard          = 0.5,                     # λ
    mu_prox              = 0.01,                     # μ (proximal)
    beta_prox            = 0.1,                      # β (trọng số proximal trong loss tổng)
    quick_eval_size      = 10_000,
    eval_batch_size      = 16384,
    log_every            = 200,                      # heartbeat: in tiến độ mỗi N batch (client lớn). 0 = tắt
    seed                 = 42,
    use_amp              = False,                    # DNN nhỏ: AMP gần như không lợi, KL fp16 dễ kém ổn định
    use_multi_gpu        = True,
    max_samples_per_client = None,                  # None = full data. Đặt int để debug nhanh.
    max_debug_batches    = None,                    # None = full. Đặt int nhỏ cho smoke test.
    best_metric          = "weighted_f1",           # tiêu chí chọn best model
    run_id               = "fdids_dnn_run1",
    local_mirror         = None,                     # path copy artifact về local/Drive (None = bỏ qua)
    # ----- resume -----
    resume               = False,
    resume_round         = 0,                        # round đã hoàn tất (load global_round_{resume_round}.pt)
)

# --------- Seed ---------
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(CFG.seed)
torch.backends.cudnn.benchmark = True

# --------- Device & GPU info ---------
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
GPU_IDS = list(range(min(N_GPU, 2))) if (CFG.use_multi_gpu and N_GPU >= 2) else ([0] if N_GPU >= 1 else [])
print(f"torch={torch.__version__}  cuda_available={torch.cuda.is_available()}  device_count={N_GPU}", flush=True)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name}  mem={p.total_memory/1e9:.1f} GB", flush=True)
print(f"GPU_IDS dùng cho client groups: {GPU_IDS}", flush=True)

# --------- Output dirs ---------
OUTPUT_ROOT    = Path("/kaggle/working/fd_ids_outputs") / CFG.run_id
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
REPORT_DIR     = OUTPUT_ROOT / "reports"
FIG_DIR        = OUTPUT_ROOT / "figures"
for d in (CHECKPOINT_DIR, REPORT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)
METRICS_CSV = REPORT_DIR / "metrics_round.csv"
print(f"DATA_DIR    = {DATA_DIR}", flush=True)
print(f"OUTPUT_ROOT = {OUTPUT_ROOT}", flush=True)


In [ ]:
# ===== Cell 2: Label mapping + Validate schema =====
CLIENT_FILES = [DATA_DIR / f"client_{i}_train.csv" for i in range(1, CFG.num_clients + 1)]
TEST_FILE    = DATA_DIR / "global_test_data.csv"
MAPPING_FILE = DATA_DIR / "label_mapping.csv"

def load_label_mapping(path):
    """Trả về (label_to_id, id_to_label). Linh hoạt với thứ tự/cấu trúc cột."""
    df = pd.read_csv(path)
    cols = list(df.columns)
    # bỏ cột index thừa nếu có
    if cols and cols[0].lower() in ("unnamed: 0", "index"):
        df = df.drop(columns=[cols[0]]); cols = list(df.columns)
    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    str_cols = [c for c in cols if not pd.api.types.is_numeric_dtype(df[c])]
    if num_cols and str_cols:
        id_col, name_col = num_cols[0], str_cols[0]
    elif len(cols) >= 2:
        # cả hai numeric: giả định [id, name] hoặc index là name
        id_col, name_col = cols[0], cols[1]
    else:
        raise ValueError(f"label_mapping.csv không đúng định dạng: cols={cols}")
    label_to_id = {str(n): int(i) for n, i in zip(df[name_col], df[id_col])}
    id_to_label = {int(i): str(n) for n, i in zip(df[name_col], df[id_col])}
    return label_to_id, id_to_label

FEATURE_NAME_HINTS = ()  # không cần; ta suy ra cột nhãn
def infer_label_column(columns):
    for cand in ("label", "Label", "LABEL", "class", "Class", "target", "y", "attack", "Attack"):
        if cand in columns:
            return cand
    return columns[-1]  # mặc định: cột cuối

assert MAPPING_FILE.exists(), f"Thiếu {MAPPING_FILE}"
label_to_id, id_to_label = load_label_mapping(MAPPING_FILE)
print(f"Số nhãn trong mapping: {len(id_to_label)}", flush=True)
assert len(id_to_label) == CFG.num_classes, f"Mapping có {len(id_to_label)} nhãn, kỳ vọng {CFG.num_classes}"

# Validate header + đếm dòng từng client (đọc nhẹ)
hdr = pd.read_csv(CLIENT_FILES[0], nrows=0).columns.tolist()
LABEL_COL = infer_label_column(hdr)
FEATURE_COLS = [c for c in hdr if c != LABEL_COL]
print(f"Cột nhãn  : {LABEL_COL}", flush=True)
print(f"Số feature: {len(FEATURE_COLS)} (kỳ vọng {CFG.input_dim})", flush=True)
assert len(FEATURE_COLS) == CFG.input_dim, f"Có {len(FEATURE_COLS)} feature, kỳ vọng {CFG.input_dim}"

missing = [f for f in CLIENT_FILES + [TEST_FILE] if not f.exists()]
assert not missing, f"Thiếu file: {missing}"
print("\\nĐủ 12 file. Số mẫu từng client sẽ in ở Cell 3 (tránh đọc toàn bộ CSV 2 lần).", flush=True)


In [ ]:
# ===== Cell 3: Load dữ liệu vào RAM 1 lần (tái dùng qua các round) =====
def _label_series_to_long(s):
    if pd.api.types.is_numeric_dtype(s):
        return torch.as_tensor(s.to_numpy(dtype=np.int64), dtype=torch.long)
    ids = s.astype(str).map(label_to_id)
    if ids.isna().any():
        bad = sorted(set(s.astype(str)[ids.isna()]))[:10]
        raise ValueError(f"Nhãn không có trong mapping: {bad}")
    return torch.as_tensor(ids.to_numpy(dtype=np.int64), dtype=torch.long)

def load_csv_tensors(path, label_col, feature_cols):
    dtypes = {c: "float32" for c in feature_cols}
    df = pd.read_csv(path, dtype=dtypes)
    X = torch.from_numpy(df[feature_cols].to_numpy(dtype=np.float32))
    y = _label_series_to_long(df[label_col])
    del df; gc.collect()
    # vệ sinh NaN/Inf (an toàn, dữ liệu lẽ ra đã sạch)
    if not torch.isfinite(X).all():
        X = torch.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    # nhãn phải nằm trong [0, num_classes-1] (bắt sớm lỗi nhãn 1-based / num_classes sai)
    if y.numel() and (int(y.min()) < 0 or int(y.max()) >= CFG.num_classes):
        raise ValueError(
            f"{Path(path).name}: nhãn ngoài [0,{CFG.num_classes-1}] "
            f"(min={int(y.min())}, max={int(y.max())}). "
            f"Kiểm tra label_mapping.csv (có thể đang 1-based) hoặc CFG.num_classes.")
    return X, y

print("Đang load 10 client vào RAM (float32)...", flush=True)
CLIENT_DATA = {}     # id -> (X_cpu, y_cpu)
CLIENT_COUNTS = {}   # id -> n (số mẫu thực dùng cho aggregation)
t0 = time.time()
for i, f in enumerate(CLIENT_FILES, 1):
    X, y = load_csv_tensors(f, LABEL_COL, FEATURE_COLS)
    CLIENT_DATA[i] = (X, y); CLIENT_COUNTS[i] = X.shape[0]
    print(f"  client {i:2d}: X={tuple(X.shape)} y={tuple(y.shape)} "
          f"({X.numel()*4/1e6:.0f} MB)  [{time.time()-t0:.0f}s]", flush=True)
print(f"Tổng mẫu train load được: {sum(CLIENT_COUNTS.values()):,}", flush=True)

print("\\nĐang load global_test_data.csv...", flush=True)
X_test, y_test = load_csv_tensors(TEST_FILE, LABEL_COL, FEATURE_COLS)
print(f"  test: X={tuple(X_test.shape)} y={tuple(y_test.shape)}", flush=True)

# --------- Quick-eval sample 10k (stratified, cố định seed) ---------
from sklearn.model_selection import train_test_split
y_test_np = y_test.numpy()
idx_all = np.arange(len(y_test_np))
qsize = min(CFG.quick_eval_size, len(idx_all))
try:
    _, quick_idx = train_test_split(idx_all, test_size=qsize,
                                    stratify=y_test_np, random_state=CFG.seed)
    sampling = "stratified"
except ValueError:
    rng = np.random.default_rng(CFG.seed)
    quick_idx = rng.choice(idx_all, size=qsize, replace=False)
    sampling = "random (fallback: vài lớp quá hiếm)"
quick_idx = np.sort(quick_idx)
X_quick = X_test[quick_idx].contiguous()
y_quick = y_test[quick_idx].contiguous()
print(f"\\nQuick-eval set: {X_quick.shape[0]:,} mẫu ({sampling})", flush=True)
uq, cnt = np.unique(y_quick.numpy(), return_counts=True)
print(f"  số lớp xuất hiện trong quick-eval: {len(uq)}/{CFG.num_classes}", flush=True)
pd.DataFrame({"feature_"+str(j): X_quick[:, j].numpy() for j in range(CFG.input_dim)}
             ).assign(label=y_quick.numpy()).to_csv(
    OUTPUT_ROOT / "quick_eval_10000.csv", index=False)
print(f"  đã lưu {OUTPUT_ROOT/'quick_eval_10000.csv'}", flush=True)


In [ ]:
# ===== Cell 4: DNN model (FD-IDS) + smoke test forward =====
class FDIDS_DNN(nn.Module):
    """DNN nhẹ theo paper: input -> [32,64,128,64,32] (ReLU) -> num_classes (logits)."""
    def __init__(self, input_dim, hidden_dims, num_classes, dropout=0.0):
        super().__init__()
        dims = [input_dim] + list(hidden_dims)
        layers = []
        for a, b in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(a, b), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Linear(dims[-1], num_classes)
    def forward(self, x):
        return self.classifier(self.features(x))   # logits (KHÔNG softmax)

def build_model(cfg=CFG):
    return FDIDS_DNN(cfg.input_dim, cfg.hidden_dims, cfg.num_classes, cfg.dropout)

_m = build_model()
_nparam = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"Số tham số model: {_nparam:,}  (~{_nparam*4/1e6:.3f} MB)", flush=True)
with torch.no_grad():
    _xb = X_quick[:8]
    _logits = _m(_xb)
print(f"logits.shape = {tuple(_logits.shape)} (kỳ vọng (8, {CFG.num_classes}))", flush=True)
assert _logits.shape == (8, CFG.num_classes)
assert torch.isfinite(_logits).all(), "logits có NaN/Inf!"
del _m, _logits; print("Forward smoke test OK", flush=True)


In [ ]:
# ===== Cell 5: Loss KD + FedProx + smoke test =====
def kd_loss(student_logits, teacher_logits, labels, T, lambda_hard):
    """L = λ·CE(student,y) + (1-λ)·T^2·KL(softmax(teacher/T) || softmax(student/T))."""
    L_hard = F.cross_entropy(student_logits, labels)
    L_soft = (T * T) * F.kl_div(
        F.log_softmax(student_logits / T, dim=1),   # log Q (student)
        F.softmax(teacher_logits / T, dim=1),       # P (teacher)
        reduction="batchmean",
    )
    return lambda_hard * L_hard + (1.0 - lambda_hard) * L_soft, L_hard.detach(), L_soft.detach()

def proximal_term(student_model, global_ref, mu):
    """L_proximal = (μ/2) Σ ||w_student - w_global||^2  (fp32)."""
    acc = 0.0
    for name, p in student_model.named_parameters():
        acc = acc + ((p - global_ref[name]) ** 2).sum()
    return (mu / 2.0) * acc

# ---- smoke test: 1 batch forward+backward, kiểm tra hữu hạn ----
_dev = DEVICE
_teacher = build_model().to(_dev); _teacher.eval()
for _p in _teacher.parameters():
    _p.requires_grad_(False)
_student = build_model().to(_dev); _student.train()
_student.load_state_dict(_teacher.state_dict())     # student khởi tạo = teacher (global)
_ref = {n: p.detach().clone() for n, p in _student.named_parameters()}
_opt = torch.optim.Adam(_student.parameters(), lr=CFG.learning_rate)
_xb = X_quick[:256].to(_dev); _yb = y_quick[:256].to(_dev)
with torch.no_grad():
    _t = _teacher(_xb)
_s = _student(_xb)
_loss_kd, _lh, _ls = kd_loss(_s, _t, _yb, CFG.temperature, CFG.lambda_hard)
_prox = CFG.beta_prox * proximal_term(_student, _ref, CFG.mu_prox)
_loss = _loss_kd + _prox
_opt.zero_grad(); _loss.backward(); _opt.step()
print(f"L_hard={_lh.item():.4f}  L_soft={_ls.item():.4f}  "
      f"L_prox={float(_prox):.6f}  L_total={_loss.item():.4f}", flush=True)
_grad_ok = all(torch.isfinite(p.grad).all() for p in _student.parameters() if p.grad is not None)
assert math.isfinite(_loss.item()) and _grad_ok, "Loss/grad không hữu hạn!"
del _teacher, _student, _opt, _ref, _t, _s, _loss
torch.cuda.empty_cache()
print("KD + FedProx smoke test OK", flush=True)


In [ ]:
# ===== Cell 6: train_one_client + debug 1 client =====
def train_one_client(client_id, global_state_cpu, X_cpu, y_cpu, device, cfg=CFG):
    """Train local 1 client với KD (teacher=global) + FedProx. Trả về (state_cpu, n_used, avg_loss, sec)."""
    t0 = time.time()
    teacher = build_model(cfg).to(device); teacher.load_state_dict(global_state_cpu); teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    student = build_model(cfg).to(device); student.load_state_dict(global_state_cpu); student.train()
    global_ref = {n: p.detach().clone() for n, p in student.named_parameters()}
    opt = torch.optim.Adam(student.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=cfg.use_amp)   # torch>=2.1
    except (TypeError, AttributeError):
        scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_amp)      # fallback

    n = X_cpu.shape[0]
    if cfg.max_samples_per_client is not None and n > cfg.max_samples_per_client:
        sub = torch.randperm(n)[: cfg.max_samples_per_client]
        Xc, yc = X_cpu[sub], y_cpu[sub]
        n = Xc.shape[0]
    else:
        Xc, yc = X_cpu, y_cpu

    nbatches = (n + cfg.batch_size - 1) // cfg.batch_size
    running, nb = 0.0, 0
    for _ in range(cfg.local_epochs):
        perm = torch.randperm(n)
        for bi, start in enumerate(range(0, n, cfg.batch_size)):
            if cfg.max_debug_batches is not None and bi >= cfg.max_debug_batches:
                break
            idx = perm[start: start + cfg.batch_size]
            xb = Xc[idx].to(device, non_blocking=True)
            yb = yc[idx].to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=cfg.use_amp):
                with torch.no_grad():
                    t_logits = teacher(xb)
                s_logits = student(xb)
                loss_kd, _, _ = kd_loss(s_logits, t_logits, yb, cfg.temperature, cfg.lambda_hard)
            loss = loss_kd + cfg.beta_prox * proximal_term(student, global_ref, cfg.mu_prox)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            lval = float(loss.detach()); running += lval; nb += 1
            if cfg.log_every and bi % cfg.log_every == 0:
                print(f"      [{device}] client {client_id:2d} batch {bi}/{nbatches} loss={lval:.4f}", flush=True)

    state_cpu = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
    avg_loss = running / max(nb, 1)
    del teacher, student, opt, global_ref, Xc, yc
    torch.cuda.empty_cache()
    return state_cpu, n, avg_loss, time.time() - t0

# ---- debug: train thử client 1 (nhỏ) 1 round ----
_g0 = build_model().state_dict()
_g0 = {k: v.cpu() for k, v in _g0.items()}
_st, _n, _l, _sec = train_one_client(1, _g0, *CLIENT_DATA[1], DEVICE)
print(f"[debug client 1] n={_n:,}  avg_loss={_l:.4f}  time={_sec:.1f}s  "
      f"state_keys={len(_st)}", flush=True)
assert set(_st.keys()) == set(_g0.keys()), "state_dict keys lệch!"
del _g0, _st; torch.cuda.empty_cache()


In [ ]:
# ===== Cell 7: GPU groups (LPT) + train group + FedAvg + evaluate =====
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def build_gpu_groups(client_counts, gpu_ids):
    """Largest-Processing-Time: gán client lớn trước vào GPU có tải thấp nhất."""
    if len(gpu_ids) <= 1:
        return {gpu_ids[0]: list(client_counts.keys())} if gpu_ids else {}
    order = sorted(client_counts, key=lambda c: client_counts[c], reverse=True)
    load = {g: 0 for g in gpu_ids}
    groups = {g: [] for g in gpu_ids}
    for c in order:
        g = min(gpu_ids, key=lambda g: load[g])
        groups[g].append(c); load[g] += client_counts[c]
    return groups

def train_client_group(gpu_id, client_ids, global_state_cpu, cfg=CFG):
    """Train tuần tự các client của 1 GPU (chạy trong 1 thread)."""
    device = torch.device(f"cuda:{gpu_id}") if torch.cuda.is_available() else torch.device("cpu")
    out = []
    for cid in client_ids:
        X, y = CLIENT_DATA[cid]
        state, n, loss, sec = train_one_client(cid, global_state_cpu, X, y, device, cfg)
        out.append((cid, state, n, loss, sec))
        print(f"    [GPU{gpu_id}] client {cid:2d}: n={n:,} loss={loss:.4f} time={sec:.1f}s", flush=True)
    return out

def weighted_fedavg(results):
    """results: list (cid, state_cpu, n, loss, sec). Trả về global_state_cpu mới."""
    total = sum(n for _, _, n, _, _ in results)
    agg = None
    for _, state, n, _, _ in results:
        w = n / total
        if agg is None:
            agg = {k: v.detach().float() * w for k, v in state.items()}
        else:
            for k in agg:
                agg[k] += state[k].detach().float() * w
    return agg

def compute_metrics(y_true, y_pred, n_classes=None):
    # labels=range(n_classes): chấm trên ĐỦ 34 lớp (lớp vắng -> 0), để macro/weighted
    # nhất quán giữa quick-eval (10k, vài lớp hiếm vắng) và full test ở Cell 9.
    labels = list(range(n_classes)) if n_classes else None
    acc = accuracy_score(y_true, y_pred)
    mp, mr, mf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    wp, wr, wf, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average="weighted", zero_division=0)
    return dict(accuracy=acc, macro_precision=mp, macro_recall=mr, macro_f1=mf,
                weighted_precision=wp, weighted_recall=wr, weighted_f1=wf)

@torch.no_grad()
def evaluate(state_cpu, X, y, device, cfg=CFG):
    model = build_model(cfg).to(device); model.load_state_dict(state_cpu); model.eval()
    preds, tot_loss, nb = [], 0.0, 0
    for start in range(0, X.shape[0], cfg.eval_batch_size):
        xb = X[start: start + cfg.eval_batch_size].to(device)
        yb = y[start: start + cfg.eval_batch_size].to(device)
        logits = model(xb)
        tot_loss += float(F.cross_entropy(logits, yb)); nb += 1
        preds.append(logits.argmax(1).cpu())
    preds = torch.cat(preds).numpy()
    metrics = compute_metrics(y.numpy(), preds, n_classes=cfg.num_classes)
    del model; torch.cuda.empty_cache()
    return preds, tot_loss / max(nb, 1), metrics

print("Cell 7 OK: build_gpu_groups / train_client_group / weighted_fedavg / evaluate", flush=True)
print("GPU groups dự kiến:", build_gpu_groups(CLIENT_COUNTS, GPU_IDS if GPU_IDS else [0]), flush=True)


In [ ]:
# ===== Cell 8: Vòng huấn luyện chính (FedProx + round-wise KD) =====
METRIC_FIELDS = ["round", "avg_client_loss", "quick_eval_loss", "accuracy",
                 "macro_precision", "macro_recall", "macro_f1",
                 "weighted_precision", "weighted_recall", "weighted_f1",
                 "round_time_sec", "is_best"]

def _append_metrics_row(row):
    write_header = not METRICS_CSV.exists()
    pd.DataFrame([row], columns=METRIC_FIELDS).to_csv(
        METRICS_CSV, mode="a", header=write_header, index=False)

# ---------- Khởi tạo / Resume ----------
gpu_ids = GPU_IDS if GPU_IDS else [0]
if CFG.resume:
    # resume=True: BẮT BUỘC có checkpoint, KHÔNG âm thầm xóa metrics & train lại từ đầu
    _ckpt = CHECKPOINT_DIR / f"global_round_{CFG.resume_round}.pt"
    assert _ckpt.exists(), (
        f"resume=True nhưng không thấy {_ckpt}. Kiểm tra CFG.resume_round (đang {CFG.resume_round}; "
        f"phải >=1 và là round đã lưu), run_id, và /kaggle/working có còn được giữ không. "
        f"Muốn train mới hãy đặt CFG.resume=False.")
    global_state = torch.load(_ckpt, map_location="cpu")
    start_round = CFG.resume_round + 1
    if METRICS_CSV.exists():
        hist = pd.read_csv(METRICS_CSV)
        hist = hist[hist["round"] <= CFG.resume_round]   # bỏ dòng > resume_round tránh trùng
        hist.to_csv(METRICS_CSV, index=False)
    else:
        hist = pd.DataFrame(columns=METRIC_FIELDS)
    best_score = hist[CFG.best_metric].max() if len(hist) else -1.0
    best_round = int(hist.loc[hist[CFG.best_metric].idxmax(), "round"]) if len(hist) else -1
    print(f"RESUME từ round {CFG.resume_round} -> bắt đầu round {start_round}. "
          f"best {CFG.best_metric}={best_score:.4f} @round {best_round}", flush=True)
else:
    set_seed(CFG.seed)
    global_state = {k: v.cpu() for k, v in build_model().state_dict().items()}
    start_round = 1; best_score = -1.0; best_round = -1
    if METRICS_CSV.exists():
        METRICS_CSV.unlink()   # chỉ xóa khi train MỚI
    print("Bắt đầu train mới từ round 1", flush=True)

# ---------- Echo cấu hình (để log tự ghi lại setup cho báo cáo) ----------
print(f"\\n>>> TRAIN START: model=DNN{list(CFG.hidden_dims)} clients={CFG.num_clients} "
      f"rounds={CFG.comm_rounds} local_epochs={CFG.local_epochs} batch={CFG.batch_size} "
      f"lr={CFG.learning_rate}", flush=True)
print(f"    KD(T={CFG.temperature}, lambda={CFG.lambda_hard}) | FedProx(mu={CFG.mu_prox}, "
      f"beta={CFG.beta_prox}) | best_metric={CFG.best_metric} | GPUs={gpu_ids} | AMP={CFG.use_amp}", flush=True)
print(f"    tổng mẫu train = {sum(CLIENT_COUNTS.values()):,} | quick-eval = {X_quick.shape[0]:,} mẫu", flush=True)

# ---------- Loop ----------
for rnd in range(start_round, CFG.comm_rounds + 1):
    t_round = time.time()
    groups = build_gpu_groups(CLIENT_COUNTS, gpu_ids)
    print(f"\\n===== ROUND {rnd}/{CFG.comm_rounds} =====  [{time.strftime('%Y-%m-%d %H:%M:%S')}]", flush=True)
    print(f"  GPU groups (gpu_id -> clients): {dict(groups)}", flush=True)

    results = []
    if len(gpu_ids) > 1:
        with ThreadPoolExecutor(max_workers=len(gpu_ids)) as ex:
            futs = [ex.submit(train_client_group, g, groups[g], global_state, CFG) for g in gpu_ids]
            for fu in futs:
                results.extend(fu.result())
    else:
        results.extend(train_client_group(gpu_ids[0], groups[gpu_ids[0]], global_state, CFG))

    # Aggregate (weighted FedAvg)
    global_state = weighted_fedavg(results)
    avg_client_loss = float(np.mean([l for _, _, _, l, _ in results]))

    # Quick eval
    _, quick_loss, m = evaluate(global_state, X_quick, y_quick, DEVICE, CFG)
    round_time = time.time() - t_round

    # Best tracking + checkpoint
    score = m[CFG.best_metric]
    is_best = score > best_score
    torch.save(global_state, CHECKPOINT_DIR / f"global_round_{rnd}.pt")
    if is_best:
        best_score, best_round = score, rnd
        torch.save(global_state, CHECKPOINT_DIR / "best_global_model.pt")

    row = dict(round=rnd, avg_client_loss=avg_client_loss, quick_eval_loss=quick_loss,
               round_time_sec=round_time, is_best=int(is_best), **m)
    _append_metrics_row(row)

    # ----- Tóm tắt ĐẦY ĐỦ của round (in ra stdout, flush để xem live qua ngrok) -----
    print(f"  ---------- ROUND {rnd}/{CFG.comm_rounds} SUMMARY  ({round_time:.1f}s) ----------", flush=True)
    print(f"    avg_client_loss = {avg_client_loss:.4f}    quick_eval_loss = {quick_loss:.4f}", flush=True)
    print(f"    accuracy        = {m['accuracy']:.4f}", flush=True)
    print(f"    macro    P/R/F1 = {m['macro_precision']:.4f} / {m['macro_recall']:.4f} / {m['macro_f1']:.4f}", flush=True)
    print(f"    weighted P/R/F1 = {m['weighted_precision']:.4f} / {m['weighted_recall']:.4f} / {m['weighted_f1']:.4f}", flush=True)
    print(f"    >> best {CFG.best_metric} = {best_score:.4f} @round {best_round}"
          f"{'   <<< NEW BEST (đã lưu best_global_model.pt)' if is_best else ''}", flush=True)
    print(f"    checkpoint: global_round_{rnd}.pt  |  metrics -> {METRICS_CSV.name}", flush=True)

print(f"\\nXong {CFG.comm_rounds} round. Best {CFG.best_metric}={best_score:.4f} @round {best_round}", flush=True)
print(f"Metrics: {METRICS_CSV}", flush=True)


In [ ]:
# ===== Cell 9: Đánh giá cuối trên TOÀN BỘ global_test_data.csv =====
from sklearn.metrics import classification_report, confusion_matrix

best_path = CHECKPOINT_DIR / "best_global_model.pt"
assert best_path.exists(), "Chưa có best_global_model.pt (chạy Cell 8 trước)."
best_state = torch.load(best_path, map_location="cpu")
print(f"Đánh giá best model trên full test ({X_test.shape[0]:,} mẫu)...", flush=True)

t0 = time.time()
preds_full, test_loss, M = evaluate(best_state, X_test, y_test, DEVICE, CFG)
print(f"  done in {time.time()-t0:.0f}s", flush=True)
y_true = y_test.numpy()

print("\\n===== FINAL METRICS (full test) =====", flush=True)
print(f"  loss               = {test_loss:.4f}")
print(f"  accuracy           = {M['accuracy']:.4f}")
print(f"  macro    P/R/F1    = {M['macro_precision']:.4f} / {M['macro_recall']:.4f} / {M['macro_f1']:.4f}")
print(f"  weighted P/R/F1    = {M['weighted_precision']:.4f} / {M['weighted_recall']:.4f} / {M['weighted_f1']:.4f}")

# Classification report 34 lớp
labels_sorted = list(range(CFG.num_classes))
target_names = [id_to_label.get(i, str(i)) for i in labels_sorted]
report_dict = classification_report(y_true, preds_full, labels=labels_sorted,
                                     target_names=target_names, zero_division=0, output_dict=True)
pd.DataFrame(report_dict).T.to_csv(REPORT_DIR / "classification_report.csv")
print("\\n" + classification_report(y_true, preds_full, labels=labels_sorted,
                                     target_names=target_names, zero_division=0), flush=True)

# Confusion matrix
cm = confusion_matrix(y_true, preds_full, labels=labels_sorted)
np.save(REPORT_DIR / "confusion_matrix.npy", cm)

# FPR/FNR attack-vs-benign (nếu xác định được BENIGN)
benign_id = None
for k, v in id_to_label.items():
    if str(v).upper().startswith("BENIGN"):
        benign_id = int(k); break
attack_block = {}
if benign_id is not None:
    yb_true = (y_true != benign_id).astype(int)   # 1 = attack
    yb_pred = (preds_full != benign_id).astype(int)
    TP = int(((yb_true == 1) & (yb_pred == 1)).sum())
    FN = int(((yb_true == 1) & (yb_pred == 0)).sum())
    FP = int(((yb_true == 0) & (yb_pred == 1)).sum())
    TN = int(((yb_true == 0) & (yb_pred == 0)).sum())
    FPR = FP / (FP + TN) if (FP + TN) else 0.0
    FNR = FN / (FN + TP) if (FN + TP) else 0.0
    attack_block = dict(benign_id=benign_id, TP=TP, FN=FN, FP=FP, TN=TN,
                        FPR=FPR, FNR=FNR,
                        attack_detection_rate=TP / (TP + FN) if (TP + FN) else 0.0)
    print(f"\\n[Attack vs Benign] FPR={FPR:.4f}  FNR={FNR:.4f}  "
          f"detection_rate={attack_block['attack_detection_rate']:.4f}", flush=True)
else:
    print("\\nKhông tìm thấy lớp BENIGN trong mapping -> bỏ qua FPR/FNR.", flush=True)

final_report = dict(run_id=CFG.run_id, test_loss=test_loss, metrics=M,
                    attack_vs_benign=attack_block,
                    config={k: (list(v) if isinstance(v, tuple) else v)
                            for k, v in vars(CFG).items()})
with open(REPORT_DIR / "final_report.json", "w") as f:
    json.dump(final_report, f, indent=2, default=str)

# ---- results_summary.txt: tóm tắt dễ copy/dán về local ----
_hist = pd.read_csv(METRICS_CSV)
_lines = []
_lines.append(f"=== FD-IDS run: {CFG.run_id} ===")
_lines.append(f"clients={CFG.num_clients} rounds={CFG.comm_rounds} local_epochs={CFG.local_epochs} "
              f"batch={CFG.batch_size} lr={CFG.learning_rate} | T={CFG.temperature} "
              f"lambda={CFG.lambda_hard} mu={CFG.mu_prox} beta={CFG.beta_prox}")
_lines.append("\\n-- Per-round metrics --")
_lines.append(_hist.to_string(index=False))
_lines.append("\\n-- FINAL (best model trên full test) --")
_lines.append(f"loss={test_loss:.4f}  accuracy={M['accuracy']:.4f}")
_lines.append(f"macro    P/R/F1 = {M['macro_precision']:.4f}/{M['macro_recall']:.4f}/{M['macro_f1']:.4f}")
_lines.append(f"weighted P/R/F1 = {M['weighted_precision']:.4f}/{M['weighted_recall']:.4f}/{M['weighted_f1']:.4f}")
if attack_block:
    _lines.append(f"attack-vs-benign: FPR={attack_block['FPR']:.4f} FNR={attack_block['FNR']:.4f} "
                  f"detection_rate={attack_block['attack_detection_rate']:.4f}")
_summary = "\\n".join(_lines)
(REPORT_DIR / "results_summary.txt").write_text(_summary)
print("\\n" + _summary, flush=True)
print(f"\\nĐã lưu báo cáo cuối vào {REPORT_DIR}", flush=True)


In [ ]:
# ===== Cell 10: Vẽ curve + confusion matrix =====
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

hist = pd.read_csv(METRICS_CSV)

# Loss curve
plt.figure(figsize=(7, 5))
plt.plot(hist["round"], hist["avg_client_loss"], "o-", label="avg_client_loss")
plt.plot(hist["round"], hist["quick_eval_loss"], "s-", label="quick_eval_loss")
plt.xlabel("Communication round"); plt.ylabel("Loss"); plt.title("Loss curve"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "loss_curve.png", dpi=120); plt.close()

# Accuracy curve
plt.figure(figsize=(7, 5))
plt.plot(hist["round"], hist["accuracy"], "o-", color="green", label="accuracy")
plt.xlabel("Communication round"); plt.ylabel("Accuracy"); plt.title("Accuracy curve"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "accuracy_curve.png", dpi=120); plt.close()

# F1 curve (macro + weighted)
plt.figure(figsize=(7, 5))
plt.plot(hist["round"], hist["macro_f1"], "o-", label="macro_f1")
plt.plot(hist["round"], hist["weighted_f1"], "s-", label="weighted_f1")
plt.xlabel("Communication round"); plt.ylabel("F1"); plt.title("F1 curve"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "f1_curve.png", dpi=120); plt.close()

# Combined
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
ax[0].plot(hist["round"], hist["avg_client_loss"], "o-", label="client")
ax[0].plot(hist["round"], hist["quick_eval_loss"], "s-", label="quick eval")
ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].plot(hist["round"], hist["accuracy"], "o-", color="green")
ax[1].set_title("Accuracy"); ax[1].grid(True, alpha=0.3)
ax[2].plot(hist["round"], hist["macro_f1"], "o-", label="macro")
ax[2].plot(hist["round"], hist["weighted_f1"], "s-", label="weighted")
ax[2].set_title("F1"); ax[2].legend(); ax[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=120); plt.close()

# Confusion matrix (normalized, log-friendly)
cm = np.load(REPORT_DIR / "confusion_matrix.npy")
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
plt.figure(figsize=(13, 11))
plt.imshow(cm_norm, cmap="viridis", aspect="auto")
plt.colorbar(label="row-normalized")
plt.title("Confusion matrix (normalized by true label)")
plt.xlabel("Predicted"); plt.ylabel("True")
tick = list(range(CFG.num_classes))
names = [id_to_label.get(i, str(i)) for i in tick]
plt.xticks(tick, names, rotation=90, fontsize=6)
plt.yticks(tick, names, fontsize=6)
plt.tight_layout(); plt.savefig(FIG_DIR / "confusion_matrix.png", dpi=130); plt.close()

print("Đã lưu figures:", flush=True)
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f}", flush=True)
print("\\nXem nhanh bảng metrics theo round:", flush=True)
print(hist.to_string(index=False), flush=True)

# ---- Mirror toàn bộ output về local/Drive nếu CFG.local_mirror được set ----
if CFG.local_mirror:
    import shutil
    dst = Path(CFG.local_mirror) / CFG.run_id
    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(OUTPUT_ROOT, dst)
        print(f"\\nĐã mirror artifact về: {dst}", flush=True)
    except Exception as e:
        print(f"\\n[mirror] bỏ qua ({e}). Tự download {OUTPUT_ROOT} từ Kaggle nếu cần.", flush=True)
else:
    print(f"\\n(Không set CFG.local_mirror) — download artifact từ: {OUTPUT_ROOT}", flush=True)


## Chiến lược test & chỉnh tham số trong quá trình train

### A. Smoke test trước khi train full (bắt buộc)
Ở **Cell 1**, đặt tạm:
```python
CFG.comm_rounds = 1
CFG.max_debug_batches = 10     # mỗi client chỉ 10 batch
CFG.quick_eval_size = 1000
```
Chạy Cell 1→8. Kiểm tra: state_dict hợp lệ, loss không NaN, `logits.shape=(B,34)`, ghi được 1 dòng `metrics_round.csv`, lưu được checkpoint.
Sau khi OK, trả lại: `comm_rounds=10`, `max_debug_batches=None`, `quick_eval_size=10_000` rồi chạy lại Cell 8.

### B. Theo dõi khi train (qua Ngrok)
Mỗi round in: `client_loss`, `quick_loss`, `accuracy`, `macro_f1`, `weighted_f1`, thời gian round, cờ `BEST`.
Dấu hiệu cần can thiệp:
- `quick_loss` tăng / dao động mạnh → giảm `learning_rate` (1e-3 → 5e-4) hoặc giảm `batch_size`.
- `accuracy`/`weighted_f1` bão hòa sớm → tăng `comm_rounds` hoặc `local_epochs`.
- Round quá lâu → tăng `batch_size` (DNN nhẹ chịu được 8192) hoặc đặt `max_samples_per_client` để cap mẫu/round.
- OOM → giảm `batch_size`, hoặc `eval_batch_size`.

### C. Các "núm vặn" theo bài báo (giữ đúng FD-IDS)
| Tham số | Mặc định | Ảnh hưởng |
| --- | --- | --- |
| `temperature` (T) | 3.0 | T lớn → soft label mượt hơn; paper thấy T=3, λ=0.5 tốt nhất |
| `lambda_hard` (λ) | 0.5 | λ→1: thiên hard-label; λ→0: thiên distillation |
| `mu_prox` (μ) | 0.01 | μ lớn → ràng buộc local sát global hơn (giảm drift) |
| `beta_prox` (β) | 0.1 | trọng số proximal trong loss tổng |
| `learning_rate` | 1e-3 | Adam |
| `batch_size` | 4096 | throughput vs số bước cập nhật |
| `comm_rounds` | 10 | số vòng FL |
| `hidden_dims` | (32,64,128,64,32) | năng lực DNN |

### D. Ablation nhanh (tùy chọn, để so sánh như paper)
- **Tắt KD**: đặt `lambda_hard=1.0` (chỉ còn CE + proximal) → so với bản đầy đủ.
- **Tắt FedProx**: đặt `beta_prox=0.0` (về FedAvg + KD).
- **FedAvg thuần**: `lambda_hard=1.0` và `beta_prox=0.0`.
Mỗi lần đổi `CFG.run_id` để tách thư mục output, dễ so sánh.

### E. Resume khi Kaggle/Ngrok bị ngắt
Đặt ở Cell 1:
```python
CFG.resume = True
CFG.resume_round = <round cuối đã lưu checkpoint>
```
Cell 8 sẽ load `global_round_{resume_round}.pt` và train tiếp, giữ nguyên `metrics_round.csv`.

### F. Artifact xuất ra `/kaggle/working/fd_ids_outputs/<run_id>/`
```
checkpoints/best_global_model.pt, global_round_1..10.pt
reports/metrics_round.csv, classification_report.csv, confusion_matrix.npy, final_report.json, results_summary.txt
figures/loss_curve.png, accuracy_curve.png, f1_curve.png, training_curves.png, confusion_matrix.png
quick_eval_10000.csv
```

### G. Lưu kết quả về local
- Mỗi round in metric ra stdout (flush) → xem trực tiếp qua ngrok.
- Cuối phiên in `results_summary.txt` (per-round + final) dễ copy/dán về local.
- Muốn notebook tự copy artifact: đặt `CFG.local_mirror="<path Drive/đồng-bộ>"` (Cell 1).
- Nếu không, **download** thư mục output từ Kaggle hoặc dán log cho Claude lưu vào `paper1/results/`.